# Example: 

This notebook follows on from the previous one, except that the following (linear) evolutive solves **will not** solve the GS equation at each timestep. 

This mode uses the linearisations from the initial equilibrium configuration at time $t=0$ (i.e. the Jacobians inside the evolutive solver) to evolve plasma shape parameters of interest (e.g. midplane radii, X-point positions) forward in time rapidly.

This can be useful for doing very fast scans forward in time into how certain shape parameters will evolve (linearly at least). Note, however, the equilibria for these simulations cannot be visualised as GS has not been solved!

The simulation proceeds in almost exactly the same way as the previous notebook so please do check that out first. 

### Static forward simulation

Begin by building the initial plasma configuration as usual. 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# build machine
from freegsnke import build_machine
tokamak = build_machine.tokamak(
    active_coils_path=f"../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
    passive_coils_path=f"../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
    limiter_path=f"../machine_configs/MAST-U/MAST-U_like_limiter.pickle",
    wall_path=f"../machine_configs/MAST-U/MAST-U_like_wall.pickle",
)

# initialise equilibrium object
from freegsnke import equilibrium_update
eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,
    Rmin=0.1, Rmax=2.0,   # radial range
    Zmin=-2.2, Zmax=2.2,  # vertical range
    nx=65,                # number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
    ny=129,               # number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
    # psi=plasma_psi
)  

# initialise profile object
from freegsnke.jtor_update import ConstrainPaxisIp
profiles = ConstrainPaxisIp(
    eq=eq,
    paxis=8.1e3,
    Ip=6.2e5,
    fvac=0.5,
    alpha_m=1.8,
    alpha_n=1.2
)

# initialise solver
from freegsnke import GSstaticsolver
GSStaticSolver = GSstaticsolver.NKGSsolver(eq)    

# set coil currents
import pickle
with open('data/simple_diverted_currents_PaxisIp.pk', 'rb') as f:
    current_values = pickle.load(f)

for key in current_values.keys():
    eq.tokamak[key].current = current_values[key]
eq.tokamak["P6"].current += 100

# carry out forward solve
GSStaticSolver.solve(eq=eq, 
                     profiles=profiles, 
                     constrain=None, 
                     target_relative_tolerance=1e-9)

# plot the resulting equilbrium
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()

### Define the plasma descriptors

Here we define the "plasma descriptors" (i.e. the shape parameters) that we want to track over time. These descriptors will be evolved according to the following equation:

$$ 
    \vec{s}(t) = \vec{s}(0) + \frac{\partial \vec{s}}{\partial \vec{I}_e} (\vec{I}_e(t) - \vec{I}_e(0)) + \frac{\partial \vec{s}}{\partial \vec{\theta}} (\vec{\theta}(t) - \vec{\theta}(0))
$$

where
 - $\vec{s}(t)$ are the plasma descriptors (defined in the following cells) approximated via the linearisation, and $\vec{s}(0)$ are the values defined by the initial equilibrium configuration. 
 - $\vec{I}_e(t) = (\vec{I}_m(t), I_p(t))$ is a vector of the currents in the metals and total plasma current at time $t$. 
 - $\vec{\theta}(t)$ are the (plasma current density) profile parameters at time $t$. 
 - $\frac{\partial \vec{s}}{\partial \vec{I}_e}$ and $\frac{\partial \vec{s}}{\partial \vec{\theta}}$ are the Jacobians calculated wrt the initial equilibrium. 

The currents come from solving the circuit equations the same way as before while the plasma profile parameters are inputs provided by the user. 

This enables a rapid (approximate) evolution of these descriptors over short periods of time - noting that the Jacobians lose accuracy as they diverge from the equilibrium from which they constructed.

The plasma descriptors are defined via a function that must take a FreeGSNKE equilibrium object as it's sole input. 

For example, below we define a function that will return an array containing: the average $Z$ position of the $J_{\phi}$ map; the radial and vertical position of the lower X-point; and the inboard midplane radius. 

This can be customsied to add any descriptors of interest to you. 

In [ ]:
# define the descriptors function (it should return an array of values and take in an eq object)
def plasma_descriptors(eq):

    # find lower X-point
    # define a "box" in which to search for the lower X-point
    XPT_BOX = [[0.33, -0.88], [0.95, -1.38]]

    # mask those points
    xpt_mask = (
        (eq.xpt[:, 0] >= XPT_BOX[0][0])
        & (eq.xpt[:, 0] <= XPT_BOX[1][0])
        & (eq.xpt[:, 1] <= XPT_BOX[0][1])
        & (eq.xpt[:, 1] >= XPT_BOX[1][1])
    )
    xpts = eq.xpt[xpt_mask, 0:2].squeeze()
    if xpts.ndim > 1 and xpts.shape[0] > 1:
        opt = eq.opt[0, 0:2]
        dists = np.linalg.norm(xpts - opt, axis=1)
        idx = np.argmin(dists)  # index of closest point
        Rx, Zx = xpts[idx, :]
    else:
        Rx, Zx = xpts

    # find avg. Z position of jtor
    Zcurrent = eq.Zcurrent()

    # inboard midplane radius
    Rin = eq.innerOuterSeparatrix()[0]

    return np.array([Zcurrent, Rx, Zx, Rin])

### Time evolution

Having defined the plasma descriptors, we can now instantiate the evolutive solver object. By including the `plasma_descriptors` function as an argument, the relevant Jacobians will be calculated to enable this evolution. 

In [ ]:
from freegsnke import nonlinear_solve

stepping = nonlinear_solve.nl_solver(
    eq=eq, 
    profiles=profiles, 
    GSStaticSolver=GSStaticSolver,
    full_timestep=5e-4, 
    plasma_resistivity=1e-6,
    max_mode_frequency=10**2.5,
    plasma_descriptor_function=plasma_descriptors
)

### Carry out time stepping

As we did in the previous notebook, we define the number of steps we wish to take (based on the timestep chosen above) and create some storage lists. 

In [ ]:
# number of time steps to simulate
max_count = 90

# initialising some variables for iteration and logging
counter = 0
t = 0

# initialise object
stepping.initialize_from_ICs(eq, profiles)

# storage
history_times = [t]
history_currents = [stepping.currents_vec]
history_plasma_descriptors = [plasma_descriptors(eq)]

In [ ]:
# constant voltages (i.e. no control or external drive)
voltages = (stepping.vessel_currents_vec*stepping.evol_metal_curr.coil_resist)[:stepping.evol_metal_curr.n_active_coils] 

# # time-dependent plasma current density profile parameters
# alpha_m = np.tile(profiles.alpha_m, max_count+1)
# alpha_m -= (0.1 * np.sin(0.05 * np.pi * np.arange(max_count+1))) # we add some perturbation

# alpha_n = np.tile(profiles.alpha_n, max_count+1)
# alpha_n += (0.1 * np.sin(0.1 * np.pi * np.arange(max_count+1))) # we add some perturbation

# paxis = np.tile(profiles.paxis, max_count+1)
# paxis += (0.1 * np.sin(0.01 * np.pi * np.arange(max_count+1))) # we add some perturbation

We can now call the timestepper with both the `linear_only=True` and `no_GS=True` options. This ensures that a linear solve is carried out **without** solving GS at each timestep. Note that you will **not** be able to extract all of the usual equilibrium information at each timestep as the internal `eq` objects no longer contain valid GS solutions! Switch back to `no_GS=False` if that's what you need. 

This means that in this mode of evolution, only the the metal currents, the plasma current, and the plasma descriptors are evolved over time.

In [ ]:
# loop over time steps
while counter < max_count:
    print(f'Step: {counter}/{max_count-1}')
    print(f'--- t = {t:.2e}')

    # carry out the time step (feed in the voltages and profile parameters)
    stepping.nlstepper(
        active_voltage_vec=voltages,
        linear_only=True,
        no_GS=True,
        verbose=False,
    )

    # store time-advanced currents and plasma descriptors
    history_currents.append(stepping.currents_vec)
    history_plasma_descriptors.append(stepping.plasma_descriptors_vec)

    t += stepping.dt_step
    history_times.append(t)
    counter += 1

# transform lists to arrays
history_currents = np.array(history_currents)
history_times = np.array(history_times)
history_plasma_descriptors = np.array(history_plasma_descriptors)

Let us carry out the same evolution, this time with the linear solver actually solving GS at each step (to compare the difference). Note that we change the way we extract the plasma descriptors as `no_GS=False`!

In [ ]:
# re-initialising some variables for iteration and logging
counter = 0
t = 0

# re-initialise object
stepping.initialize_from_ICs(eq, profiles)

# storage
history_times_with_GS = [t]
history_currents_with_GS = [stepping.currents_vec]
history_plasma_descriptors_with_GS = [plasma_descriptors(eq)]

# loop over the time steps
while counter < max_count:
    print(f'Step: {counter}/{max_count-1}')
    print(f'--- t = {t:.2e}')
    
    # carry out the time step (feed in the voltages and profile parameters)
    stepping.nlstepper(
        active_voltage_vec=voltages,
        linear_only=True,
        no_GS=False,
        verbose=False,
    )

    # store time-advanced currents and plasma descriptors
    history_currents_with_GS.append(stepping.currents_vec)
    history_plasma_descriptors_with_GS.append(plasma_descriptors(stepping.eq1))  # <-- changed!

    t += stepping.dt_step
    history_times_with_GS.append(t)
    counter += 1

# transform lists to arrays
history_currents_with_GS = np.array(history_currents_with_GS)
history_times_with_GS = np.array(history_times_with_GS)
history_plasma_descriptors_with_GS = np.array(history_plasma_descriptors_with_GS)

Now we can compare the difference between the linear evolution with and without GS solves at each step. We see that for some parameters the evolution diverges very quickly with solving GS!

In [ ]:
fig, ax = plt.subplots(4,1, figsize=(10, 12), dpi=80)
ax = ax.flatten()

labels = ["Zcurrent [m]", "Rx [m]", "Zx [m]", "Rin [m]"]

for i in range(4):
    ax[i].plot(history_times_with_GS, history_plasma_descriptors_with_GS[:, i], 'k+', label="Linear (with GS)")
    ax[i].plot(history_times, history_plasma_descriptors[:, i], 'm3', label="Linear (without GS)")
    ax[i].set_xlabel("Time (s)")
    ax[i].set_ylabel(labels[i])
    ax[i].legend()
    ax[i].grid()

fig.tight_layout()
